In [11]:
import gymnasium as gym
import numpy as np

In [12]:
def build_env():
    env = gym.make("gymnasium_2048:gymnasium_2048/TwentyFortyEight-v0", size=4, max_pow=16)
    env.reset()
    return env

In [13]:
def features(state,action):
    feature = state.flatten()
    possible_actions = [[1,0,0,0],[0,1,0,0],[0,0,1,0],[0,0,0,1]]
    feature = np.append(feature,possible_actions[action])
    return feature

In [14]:
def features_value(state):
    return state.flatten()

In [15]:
def estimate(state,action,w_policy):
    return np.dot(w_policy.transpose(),features(state,action))

In [16]:
def policy_est(state,action,w_policy):
    numerator = np.exp(estimate(state,action,w_policy))
    denominator = 0
    for a in range(4):
        denominator += np.exp(estimate(state,a,w_policy))
    return numerator/denominator


In [17]:
def value_est(state,w_value):
    return np.dot(w_value.transpose(),features_value(state))

In [18]:
def RBL(env,episodes,epsilon,gamma,lr_policy,lr_value):
    w_policy = np.zeros(260)
    w_value = np.zeros(256)
    for _ in range(episodes):
        epsilon = max(0.05,epsilon*0.999)
        state,_ = env.reset()
        possibleactions = [0,1,2,3]
        if np.random.rand() < epsilon:
            action = np.random.choice(possibleactions)
        else :
            action = np.argmax([policy_est(state,actions,w_policy) for actions in possibleactions])
        done = False
        episode = []
        while(True):
            next_state, reward, done, _, _ = env.step(action)
            if np.random.rand()<epsilon:
                next_action = np.random.choice(possibleactions)
            else:
                next_action = np.argmax([policy_est(next_state,actions,w_policy) for actions in possibleactions])
            episode.append((state,action,reward))
            if done == True:
                break
            state = next_state
            action = next_action
        G = 0
        for t,point in enumerate(reversed(episode)):
            state,action,reward = point
            G = gamma*G + reward
            delta = G - value_est(state,w_value)
            temp = 0
            for b in range(4):
                temp+=policy_est(state,b,w_policy)*(features(state,b))
            w_value+=(gamma**(len(episode)-t-1))*(lr_value)*(delta)*(features_value(state))
            w_policy+=(gamma**(len(episode)-t-1))*(lr_policy)*(delta)*(features(state,action) - temp)
    return w_policy,w_value



In [19]:
def main():
    gamma = 0.99
    learning_rate_value = 0.01
    learning_rate_policy = 0.01
    epsilon_start = 0.7
    episodes = 1000
    env = build_env()
    w_policy,w_value = RBL(env, episodes,epsilon_start,gamma,learning_rate_policy,learning_rate_value)
    print(w_policy)
    print(w_value)
    total_reward = 0  
    for i in range(100):
        state, _ = env.reset()
        while(True):
            if np.random.rand() < 0.05:
                action = np.random.randint(0,4)
            else :
                action = np.argmax([policy_est(state, a, w_policy) for a in range(4)])
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            if terminated or truncated:
                print(f"Episode ended.{total_reward} , {i}")
                break
    print(f"Test episode total reward: {total_reward/100}")

In [20]:
if __name__ == "__main__":
    main()

[ 8.14763115e-15  4.62250355e-15  1.17600321e-14  4.09822780e-15
 -7.83360623e-16 -2.43896088e-15  8.04200890e-16  7.10653173e-17
 -8.98318805e-17  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  6.46060570e-15  3.85266079e-15 -5.19267194e-16  1.36479005e-14
  2.81635928e-15  6.14223403e-16 -3.73361653e-16 -2.85931353e-16
 -2.16819601e-17  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  1.18821003e-14  8.20909933e-15  9.03803572e-16  4.20520735e-16
  2.53668584e-15 -2.48160586e-15  5.61351809e-15 -8.85261958e-16
 -7.35262755e-18  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  1.11601262e-14  8.38524717e-15 -7.10940253e-15  8.42041222e-15
 -3.03843832e-15  7.18165740e-15  1.34355433e-15 -1.32692267e-16
 -1.89567452e-17  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000